Aiming for higher than .9 balanced accuracy over the #2 and #3 drafts

In [ ]:

# setup: imports, seeds, GPU

import os, random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

# Mixed precision
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision policy:", mixed_precision.global_policy())

AUTOTUNE = tf.data.AUTOTUNE






TF version: 2.19.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mixed precision policy: <DTypePolicy "mixed_float16">


In [ ]:
# load EuroSAT RGB from Hugging Face

!pip -q install datasets
from datasets import load_dataset

ds_hf = load_dataset("blanchon/EuroSAT_RGB")

NUM_CLASSES = ds_hf["train"].features["label"].num_classes
label_names = ds_hf["train"].features["label"].names

IMG_H, IMG_W = 64, 64
BATCH_SIZE = 256

split_sizes = {k: len(ds_hf[k]) for k in ["train", "validation", "test"]}
print("NUM_CLASSES:", NUM_CLASSES)
print("Classes:", label_names)
print("Split sizes:", split_sizes)





/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/105M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/34.8M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/34.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5400 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5400 [00:00<?, ? examples/s]

NUM_CLASSES: 10
Classes: ['Annual Crop', 'Forest', 'Herbaceous Vegetation', 'Highway', 'Industrial Buildings', 'Pasture', 'Permanent Crop', 'Residential Buildings', 'River', 'SeaLake']
Split sizes: {'train': 16200, 'validation': 5400, 'test': 5400}


In [ ]:

# build tf.data pipelines, normalize,one-hot labels

def hf_to_tfds(hf_split, training: bool):
    def gen():
        for ex in hf_split:
            img = np.array(ex["image"], dtype=np.uint8)
            y = int(ex["label"])
            yield img, y

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec(shape=(IMG_H, IMG_W, 3), dtype=tf.uint8),
            tf.TensorSpec(shape=(), dtype=tf.int32),
        ),
    )

    def preprocess(x, y):
        x = tf.cast(x, tf.float32) / 255.0
        y = tf.one_hot(y, NUM_CLASSES)
        return x, y

    ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE)

    if training:
        ds = ds.shuffle(20_000, seed=SEED, reshuffle_each_iteration=True)
        ds = ds.cache()
        ds = ds.repeat()
        ds = ds.batch(BATCH_SIZE, drop_remainder=True).prefetch(AUTOTUNE)
    else:
        ds = ds.batch(BATCH_SIZE).cache().prefetch(AUTOTUNE)

    return ds

train_ds = hf_to_tfds(ds_hf["train"], training=True)
val_ds   = hf_to_tfds(ds_hf["validation"], training=False)
test_ds  = hf_to_tfds(ds_hf["test"], training=False)

# Step counts
train_steps = int(np.ceil(split_sizes["train"] / BATCH_SIZE))       # 64
val_steps   = int(np.ceil(split_sizes["validation"] / BATCH_SIZE))  # 22
test_steps  = int(np.ceil(split_sizes["test"] / BATCH_SIZE))        # 22
print("Steps:", {"train_steps": train_steps, "val_steps": val_steps, "test_steps": test_steps})

# check batch
for xb, yb in val_ds.take(1):
    print("X:", xb.shape, xb.dtype, "| y:", yb.shape, yb.dtype)


Steps: {'train_steps': 64, 'val_steps': 22, 'test_steps': 22}
X: (256, 64, 64, 3) <dtype: 'float32'> | y: (256, 10) <dtype: 'float32'>


In [ ]:
# check labels
def label_hist_onehot(ds, n_batches=30):
    counts = np.zeros(NUM_CLASSES, dtype=int)
    seen = 0
    for xb, yb in ds.take(n_batches):
        labs = np.argmax(yb.numpy(), axis=1)
        for l in labs:
            counts[l] += 1
        seen += len(labs)
    print("Seen:", seen)
    for i, c in enumerate(counts):
        print(f"{i:2d} {label_names[i]:>22}: {c}")

label_hist_onehot(val_ds, n_batches=30)


Seen: 5400
 0            Annual Crop: 613
 1                 Forest: 605
 2  Herbaceous Vegetation: 628
 3                Highway: 499
 4   Industrial Buildings: 507
 5                Pasture: 409
 6         Permanent Crop: 481
 7  Residential Buildings: 583
 8                  River: 511
 9                SeaLake: 564


In [ ]:
# baseline model
def build_functional_cnn_noaug(input_shape=(64, 64, 3), num_classes=10):
    inputs = keras.Input(shape=input_shape, name="input")
    x = inputs

    # Block 1
    x = layers.Conv2D(32, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(32, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)

    # Block 2
    x = layers.Conv2D(64, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(64, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)

    # Block 3
    x = layers.Conv2D(128, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(128, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)

    # Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    # Mixed precision stability: float32 outputs
    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32", name="probs")(x)
    return keras.Model(inputs, outputs, name="EuroSAT_CNN_noaug")

base_model = build_functional_cnn_noaug((IMG_H, IMG_W, 3), NUM_CLASSES)
base_model.summary()


Model: "EuroSAT_CNN_noaug"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 64, 64, 32)     │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 32)     │         9,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 64)     │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 64)     │        36,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 16, 16, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 16, 16, 128)    │       147,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 8, 128)      │             

 Total params: 324,714 (1.24 MB)

 Trainable params: 323,306 (1.23 MB)

 Non-trainable params: 1,408 (5.50 KB)

In [ ]:
# train baseline model (from #3 model not submitted between earlier draft submission & this #4 notebook)
base_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=[
        "accuracy",
        keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc"),
    ],
)

base_callbacks = [
    keras.callbacks.ModelCheckpoint("baseline_best.keras", monitor="val_accuracy", save_best_only=True),
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
]

history_base = base_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    steps_per_epoch=train_steps,
    validation_steps=val_steps,
    callbacks=base_callbacks,
    verbose=1,
)


Epoch 1/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 61s 111ms/step - accuracy: 0.5876 - loss: 1.2481 - top3_acc: 0.8160 - val_accuracy: 0.1044 - val_loss: 2.4947 - val_top3_acc: 0.2937 - learning_rate: 3.0000e-04
Epoch 2/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8143 - loss: 0.5463 - top3_acc: 0.9708 - val_accuracy: 0.1044 - val_loss: 3.5584 - val_top3_acc: 0.3600 - learning_rate: 3.0000e-04
Epoch 3/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8616 - loss: 0.4141 - top3_acc: 0.9846 - val_accuracy: 0.1044 - val_loss: 4.7191 - val_top3_acc: 0.2933 - learning_rate: 3.0000e-04
Epoch 4/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8910 - loss: 0.3263 - top3_acc: 0.9891 - val_accuracy: 0.1044 - val_loss: 5.2114 - val_top3_acc: 0.3100 - learning_rate: 1.5000e-04
Epoch 5/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.8994 - loss: 0.2948 - top3_acc: 0.9918 - val_accuracy: 0.1080 - val_loss: 4.8826 - val_top3_acc: 0.3287 - learning_rate: 1.5000e-04
Epoch 6/20
64

In [ ]:
# metrics
!pip -q install scikit-learn
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix

baseline_best = keras.models.load_model("baseline_best.keras")
base_eval = baseline_best.evaluate(test_ds, steps=test_steps, verbose=1)
print("Baseline test metrics:", dict(zip(baseline_best.metrics_names, base_eval)))

def preds_and_true_onehot(model, ds, steps=None):
    y_true, y_pred = [], []
    it = ds.take(steps) if steps is not None else ds
    for xb, yb in it:
        probs = model.predict(xb, verbose=0)
        y_true.append(np.argmax(yb.numpy(), axis=1))
        y_pred.append(np.argmax(probs, axis=1))
    return np.concatenate(y_true), np.concatenate(y_pred)

y_true, y_pred = preds_and_true_onehot(baseline_best, test_ds, steps=test_steps)
print("Baseline balanced accuracy:", balanced_accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))

cm_base = confusion_matrix(y_true, y_pred)
print("Baseline confusion matrix shape:", cm_base.shape)


22/22 ━━━━━━━━━━━━━━━━━━━━ 7s 223ms/step - accuracy: 0.9380 - loss: 0.1928 - top3_acc: 0.9961
Baseline test metrics: {'loss': 0.22750027477741241, 'compile_metrics': 0.925000011920929}
Baseline balanced accuracy: 0.9225913002877977
                       precision    recall  f1-score   support

          Annual Crop       0.90      0.95      0.92       596
               Forest       0.98      0.99      0.98       608
Herbaceous Vegetation       0.86      0.93      0.89       573
              Highway       0.81      0.87      0.84       496
 Industrial Buildings       0.95      0.94      0.95       501
              Pasture       0.92      0.93      0.92       396
       Permanent Crop       0.93      0.83      0.88       538
Residential Buildings       0.98      0.96      0.97       554
                River       0.94      0.84      0.89       529
              SeaLake       0.99      0.99      0.99       609

             accuracy                           0.93      5400
          

In [ ]:
# transfer learning model definition

def build_efficientnet_tl(num_classes=10):
    inputs = keras.Input(shape=(64, 64, 3), name="input_64")

    # resize to backbone expected size
    x = layers.Resizing(224, 224, name="resize_224")(inputs)

    # augmentation
    x = layers.RandomFlip("horizontal")(x)

    # dataset is [0,1], scale back up before preprocess_input.
    x = layers.Lambda(lambda t: t * 255.0, name="to_255")(x)
    x = layers.Lambda(tf.keras.applications.efficientnet_v2.preprocess_input, name="effv2_preprocess")(x)

    backbone = keras.applications.EfficientNetV2B0(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3),
    )
    backbone.trainable = False  # Stage 1: freeze

    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.25)(x)

    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)
    return keras.Model(inputs, outputs, name="EuroSAT_EfficientNetV2B0_TL")



In [ ]:
# train with frozen backbone

# Rebuild TL model fresh
tl_model = build_efficientnet_tl(NUM_CLASSES)

tl_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.02),
    metrics=[
        keras.metrics.CategoricalAccuracy(name="accuracy"),
        keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc"),
    ],
)

tl_stage1_callbacks = [
    keras.callbacks.ModelCheckpoint("tl_stage1_best.keras", monitor="val_accuracy", save_best_only=True),
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
]

history1 = tl_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    steps_per_epoch=train_steps,
    validation_steps=val_steps,
    callbacks=tl_stage1_callbacks,
    verbose=1,
)

print("Best stage1 val_accuracy:", max(history1.history["val_accuracy"]))



Epoch 1/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 104s 750ms/step - accuracy: 0.5877 - loss: 1.3822 - top3_acc: 0.7996 - val_accuracy: 0.8841 - val_loss: 0.8142 - val_top3_acc: 0.9839 - learning_rate: 0.0010
Epoch 2/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 3s 41ms/step - accuracy: 0.9028 - loss: 0.4341 - top3_acc: 0.9892 - val_accuracy: 0.9272 - val_loss: 0.5629 - val_top3_acc: 0.9924 - learning_rate: 0.0010
Epoch 3/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 3s 41ms/step - accuracy: 0.9257 - loss: 0.3737 - top3_acc: 0.9918 - val_accuracy: 0.9419 - val_loss: 0.4217 - val_top3_acc: 0.9952 - learning_rate: 0.0010
Epoch 4/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - accuracy: 0.9358 - loss: 0.3411 - top3_acc: 0.9952 - val_accuracy: 0.9533 - val_loss: 0.3434 - val_top3_acc: 0.9969 - learning_rate: 0.0010
Epoch 5/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - accuracy: 0.9438 - loss: 0.3232 - top3_acc: 0.9966 - val_accuracy: 0.9561 - val_loss: 0.3045 - val_top3_acc: 0.9972 - learning_rate: 0.0010
Epoch 6/10
64/64 ━━━━━━━━━━━━━━━

In [ ]:
stage1_eval = tl_model.evaluate(test_ds, steps=test_steps, verbose=1)
print("Stage1 test:", dict(zip(tl_model.metrics_names, stage1_eval)))


22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9654 - loss: 0.2599 - top3_acc: 0.9964
Stage1 test: {'loss': 0.26715055108070374, 'compile_metrics': 0.9612963199615479}


In [ ]:
# side-by-side metrics

import numpy as np
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix

def eval_to_dict(model, ds, steps, verbose=0):
    """Evaluate and return a dict of metric_name -> value.
    Handles Keras 3 'compile_metrics' naming by falling back to accuracy in results.
    """
    res = model.evaluate(ds, steps=steps, verbose=verbose)
    names = model.metrics_names
    d = dict(zip(names, res))

    if "accuracy" not in d and len(res) >= 2:
        d["accuracy"] = res[1]
    if "top3_acc" not in d:

        if len(res) >= 3:
            d["top3_acc"] = res[2]
    return d

def preds_and_true_onehot(model, ds, steps):
    """Return y_true (int labels) and y_pred (int labels) for one-hot datasets."""
    y_true, y_pred = [], []
    for xb, yb in ds.take(steps):
        probs = model.predict(xb, verbose=0)
        y_true.append(np.argmax(yb.numpy(), axis=1))
        y_pred.append(np.argmax(probs, axis=1))
    return np.concatenate(y_true), np.concatenate(y_pred)

def full_report(model, ds, steps, label_names):
    """Compute accuracy, balanced accuracy, and classification report."""
    y_true, y_pred = preds_and_true_onehot(model, ds, steps)
    bal = balanced_accuracy_score(y_true, y_pred)
    rep = classification_report(y_true, y_pred, target_names=label_names, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    return bal, rep, cm


In [ ]:

#  Evaluate both models on test
#    - baseline_best:   baseline checkpoint
#    - tl_model:   Stage 1 transfer learning model



base_metrics = eval_to_dict(baseline_best, test_ds, test_steps, verbose=1)
tl_metrics   = eval_to_dict(tl_model,       test_ds, test_steps, verbose=1)

print("Baseline metrics:", base_metrics)
print("TL Stage1 metrics:", tl_metrics)


22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9380 - loss: 0.1928 - top3_acc: 0.9961
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9654 - loss: 0.2599 - top3_acc: 0.9964
Baseline metrics: {'loss': 0.22750027477741241, 'compile_metrics': 0.925000011920929, 'accuracy': 0.925000011920929, 'top3_acc': 0.9951851963996887}
TL Stage1 metrics: {'loss': 0.26715055108070374, 'compile_metrics': 0.9612963199615479, 'accuracy': 0.9612963199615479, 'top3_acc': 0.997592568397522}


In [ ]:
# balanced accuracy report
base_bal, base_rep, base_cm = full_report(baseline_best, test_ds, test_steps, label_names)
tl_bal,   tl_rep,   tl_cm   = full_report(tl_model,       test_ds, test_steps, label_names)

print("\nBASELINE balanced accuracy:", base_bal)
print(base_rep)

print("\nTL Stage1 balanced accuracy:", tl_bal)
print(tl_rep)

print("\nConfusion matrices shapes:", base_cm.shape, tl_cm.shape)



BASELINE balanced accuracy: 0.9225913002877977
                       precision    recall  f1-score   support

          Annual Crop       0.90      0.95      0.92       596
               Forest       0.98      0.99      0.98       608
Herbaceous Vegetation       0.86      0.93      0.89       573
              Highway       0.81      0.87      0.84       496
 Industrial Buildings       0.95      0.94      0.95       501
              Pasture       0.92      0.93      0.92       396
       Permanent Crop       0.93      0.83      0.88       538
Residential Buildings       0.98      0.96      0.97       554
                River       0.94      0.84      0.89       529
              SeaLake       0.99      0.99      0.99       609

             accuracy                           0.93      5400
            macro avg       0.92      0.92      0.92      5400
         weighted avg       0.93      0.93      0.92      5400


TL Stage1 balanced accuracy: 0.9604269444264457
                  

In [ ]:
# summary table
def fmt(x, nd=4):
    return None if x is None else float(np.round(x, nd))

summary = {
    "Baseline": {
        "test_loss": fmt(base_metrics.get("loss")),
        "test_acc":  fmt(base_metrics.get("accuracy")),
        "test_top3": fmt(base_metrics.get("top3_acc")),
        "bal_acc":   fmt(base_bal),
    },
    "TL_Stage1": {
        "test_loss": fmt(tl_metrics.get("loss")),
        "test_acc":  fmt(tl_metrics.get("accuracy")),
        "test_top3": fmt(tl_metrics.get("top3_acc")),
        "bal_acc":   fmt(tl_bal),
    }
}

print("\n=== Side-by-side summary ===")
for k, v in summary.items():
    print(k, "->", v)


winner = max(summary.keys(), key=lambda m: summary[m]["bal_acc"])
print("\nWinner by balanced accuracy:", winner)



=== Side-by-side summary ===
Baseline -> {'test_loss': 0.2275, 'test_acc': 0.925, 'test_top3': 0.9952, 'bal_acc': 0.9226}
TL_Stage1 -> {'test_loss': 0.2672, 'test_acc': 0.9613, 'test_top3': 0.9976, 'bal_acc': 0.9604}

Winner by balanced accuracy: TL_Stage1


In [ ]:

# Identify confusions

def top_confusions(cm, label_names, k=10):
    cm2 = cm.copy()
    np.fill_diagonal(cm2, 0)
    pairs = []
    for i in range(cm2.shape[0]):
        for j in range(cm2.shape[1]):
            if cm2[i, j] > 0:
                pairs.append((cm2[i, j], label_names[i], label_names[j]))
    pairs.sort(reverse=True, key=lambda x: x[0])
    return pairs[:k]

print("\nTop confusions — Baseline:")
for c, true_lbl, pred_lbl in top_confusions(base_cm, label_names, k=10):
    print(f"{true_lbl:>22} → {pred_lbl:<22} : {c}")

print("\nTop confusions — TL Stage1:")
for c, true_lbl, pred_lbl in top_confusions(tl_cm, label_names, k=10):
    print(f"{true_lbl:>22} → {pred_lbl:<22} : {c}")



Top confusions — Baseline:
        Permanent Crop → Herbaceous Vegetation  : 48
                 River → Highway                : 40
  Industrial Buildings → Highway                : 23
        Permanent Crop → Annual Crop            : 23
                 River → Annual Crop            : 17
               Highway → Annual Crop            : 15
 Herbaceous Vegetation → Permanent Crop         : 14
               Highway → River                  : 14
               Pasture → Herbaceous Vegetation  : 13
        Permanent Crop → Highway                : 13

Top confusions — TL Stage1:
                 River → Highway                : 25
        Permanent Crop → Herbaceous Vegetation  : 20
                Forest → Herbaceous Vegetation  : 15
               Pasture → Herbaceous Vegetation  : 9
 Herbaceous Vegetation → Permanent Crop         : 8
  Industrial Buildings → Residential Buildings  : 8
               Highway → River                  : 7
               Highway → Annual Crop          